In [3]:
import os
import torch
import numpy as np
import pandas as pd

from torchvision import models, transforms
from PIL import Image, ImageFile
from collections import Counter
import urllib.request

# Fix truncated image issues
ImageFile.LOAD_TRUNCATED_IMAGES = True


# ================================
# SETTINGS
# ================================
DATA_FOLDER = "/Users/jaeeponde/Thesis/Data_100"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ================================
# TRANSFORMS
# ================================
transform_224 = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

transform_299 = transforms.Compose([
    transforms.Resize(342),
    transforms.CenterCrop(299),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])


def load_image(path, transform):
    img = Image.open(path).convert("RGB")
    return transform(img).unsqueeze(0).to(DEVICE)


# ================================
# LOAD IMAGENET LABELS
# ================================
LABELS_URL = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
classes_txt = urllib.request.urlopen(LABELS_URL).read().decode("utf-8").splitlines()


# ================================
# LOAD MODELS
# ================================
def load_all_models():
    models_dict = {}

    # ResNet50
    models_dict["ResNet50"] = (
        models.resnet50(weights=models.ResNet50_Weights.DEFAULT).to(DEVICE).eval(),
        transform_224
    )

    # MobileNetV2
    models_dict["MobileNetV2"] = (
        models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT).to(DEVICE).eval(),
        transform_224
    )

    # VGG16
    models_dict["VGG16"] = (
        models.vgg16(weights=models.VGG16_Weights.DEFAULT).to(DEVICE).eval(),
        transform_224
    )

    # InceptionV3 (FIXED)
    inception = models.inception_v3(
        weights=models.Inception_V3_Weights.DEFAULT,
        aux_logits=True   # MUST be True
    )
    inception.aux_logits = False  # disable after loading
    inception = inception.to(DEVICE).eval()

    models_dict["InceptionV3"] = (inception, transform_299)

    return models_dict


models_dict = load_all_models()


# ================================
# GET CLASS FOLDERS
# ================================
class_folders = sorted([
    d for d in os.listdir(DATA_FOLDER)
    if os.path.isdir(os.path.join(DATA_FOLDER, d))
])


# ================================
# MAIN LOOP
# ================================
results = {}

for cls in class_folders:

    print(f"\nProcessing class: {cls}")

    folder = os.path.join(DATA_FOLDER, cls)

    image_files = [
        f for f in os.listdir(folder)
        if f.lower().endswith((".jpg",".jpeg",".png"))
    ][:100]

    results[cls] = {}

    if len(image_files) == 0:
        print(f"⚠️ No images found for class {cls}")
        continue

    for model_name, (model, transform) in models_dict.items():

        preds = []
        fail_count = 0

        with torch.no_grad():
            for f in image_files:
                path = os.path.join(folder, f)

                try:
                    img = load_image(path, transform)
                    logits = model(img)

                    # FIX Inception tuple output
                    if isinstance(logits, tuple):
                        logits = logits[0]

                    pred = torch.argmax(logits, dim=1).item()
                    preds.append(pred)

                except Exception as e:
                    fail_count += 1
                    continue

        total = len(preds)

        print(f"{model_name}: {total} valid, {fail_count} failed")

        if total == 0:
            results[cls][model_name] = [("No valid predictions", 0)] * 5
            continue

        counter = Counter(preds)
        top5 = counter.most_common(5)

        while len(top5) < 5:
            top5.append((top5[-1][0], 0))

        top5_named = [
            (classes_txt[idx], round(100 * count / total, 2))
            for idx, count in top5
        ]

        results[cls][model_name] = top5_named


# ================================
# DISPLAY TABLES
# ================================
for cls in results:

    print(f"\n================ {cls} ================")

    df = pd.DataFrame()

    for model_name in results[cls]:
        entries = results[cls][model_name]
        formatted = [f"{name} ({pct}%)" for name, pct in entries]
        df[model_name] = formatted

    df.index = [f"Top {i+1}" for i in range(len(df))]

    print(df)


Processing class: Cars
ResNet50: 100 valid, 0 failed
MobileNetV2: 100 valid, 0 failed
VGG16: 100 valid, 0 failed
InceptionV3: 100 valid, 0 failed

Processing class: Cats
ResNet50: 100 valid, 0 failed
MobileNetV2: 100 valid, 0 failed
VGG16: 100 valid, 0 failed
InceptionV3: 100 valid, 0 failed

Processing class: Dogs
ResNet50: 100 valid, 0 failed
MobileNetV2: 100 valid, 0 failed
VGG16: 100 valid, 0 failed
InceptionV3: 100 valid, 0 failed

Processing class: Rangoli
ResNet50: 100 valid, 0 failed
MobileNetV2: 100 valid, 0 failed
VGG16: 100 valid, 0 failed
InceptionV3: 100 valid, 0 failed

Processing class: memes
ResNet50: 100 valid, 0 failed
MobileNetV2: 100 valid, 0 failed
VGG16: 100 valid, 0 failed
InceptionV3: 100 valid, 0 failed

Processing class: microscopy
ResNet50: 100 valid, 0 failed
MobileNetV2: 100 valid, 0 failed
VGG16: 100 valid, 0 failed
InceptionV3: 100 valid, 0 failed

================ Cars ================
                  ResNet50         MobileNetV2                VGG16 

In [4]:
df

,ResNet50,MobileNetV2,VGG16,InceptionV3
Top 1,nematode (11.0%),honeycomb (13.0%),honeycomb (9.0%),nematode (9.0%)
Top 2,strainer (8.0%),strainer (9.0%),matchstick (6.0%),honeycomb (8.0%)
Top 3,honeycomb (7.0%),jellyfish (5.0%),ping-pong ball (6.0%),Petri dish (7.0%)
Top 4,lemon (5.0%),matchstick (5.0%),strainer (6.0%),saltshaker (7.0%)
Top 5,matchstick (5.0%),lemon (5.0%),golf ball (5.0%),strainer (6.0%)
